<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/03_core/01_artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04. Artifacts: 再利用するものを版管理する

## この章のゴール
`add_file`、`add_dir`、`add_reference` で Artifact を作り、別の run から使います。

## 注目する W&B コード
```python
artifact.add_file("metrics.json")
artifact.add_dir("dataset")
artifact.add_reference("file:///...", name="references/raw-data")
run.log_artifact(artifact)
run.use_artifact("name:latest")
```

In [ ]:
# 準備セル: ローカルでは .env、Colab では Secrets から WANDB_API_KEY を読みます。
from pathlib import Path
import urllib.request

if not Path("course_utils.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py", "course_utils.py")

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb()

## Artifact に入れるものを作る
ローカルファイル、ディレクトリ、外部参照をそれぞれ用意します。`add_reference` は実データを W&B にアップロードせず、URI を Artifact の manifest に残します。実務では `s3://...` や `gs://...` などにも使います。

In [ ]:
from pathlib import Path
import json

root = Path("outputs/04_artifacts")
single_file = root / "metrics.json"
dataset_dir = root / "dataset"
external_file = root / "external_store" / "raw-notes.txt"

dataset_dir.mkdir(parents=True, exist_ok=True)
external_file.parent.mkdir(parents=True, exist_ok=True)

single_file.write_text(json.dumps({"rows": 2, "source": "demo"}, indent=2))
(dataset_dir / "samples.jsonl").write_text(
    '{"text": "sample A", "label": 1}\n'
    '{"text": "sample B", "label": 0}\n'
)
external_file.write_text("This file is tracked by URI reference.\n")

print("single_file:", single_file)
print("dataset_dir:", dataset_dir)
print("reference_uri:", external_file.resolve().as_uri())

## add_file / add_dir / add_reference をまとめて log する
`logged.wait()` で Artifact のアップロード完了を待つと、次のセルで確実に `use_artifact` できます。

In [ ]:
import wandb

artifact = wandb.Artifact(
    "toy-research-notes",
    type="dataset",
    description="Course demo dataset with file, directory, and URI reference examples.",
    metadata={"course": "models-initial-course"},
)
artifact.add_file(str(single_file), name="metrics/metrics.json")
artifact.add_dir(str(dataset_dir), name="dataset")
artifact.add_reference(external_file.resolve().as_uri(), name="references/raw-notes.txt")

with wandb.init(**WANDB_INIT, name="04-log-artifact-all-add-methods", job_type="create-artifact", tags=["course", "artifact"]) as run:
    logged = run.log_artifact(artifact, aliases=["latest", "course-demo"])
    logged.wait()
    artifact_ref = f"{run.entity}/{run.project}/toy-research-notes:latest"
    artifact_url = logged.url

print("Artifact ref:", artifact_ref)
print("Artifact URL:", artifact_url)

## 別の run から Artifact を使う
作った Artifact を別の run で読みます。これで W&B 上の lineage に「作った run」と「使った run」が残ります。

In [ ]:
with wandb.init(**WANDB_INIT, name="04-use-artifact", job_type="use-artifact", tags=["course", "artifact"]) as run:
    artifact = run.use_artifact(artifact_ref, type="dataset")
    downloaded = Path(artifact.download())
    downloaded_files = sorted(str(p.relative_to(downloaded)) for p in downloaded.rglob("*") if p.is_file())
    run.log({"artifact/downloaded_files": len(downloaded_files)})

print("Downloaded files:")
for name in downloaded_files:
    print("-", name)

## W&B App で確認すること
- Artifacts tab に `toy-research-notes` がある
- Artifact の Files に `metrics/`、`dataset/`、`references/` がある
- `references/raw-notes.txt` は URI reference として記録されている
- Lineage で作成 run から利用 run への関係を確認できる